# Midterm Discourse: K-Means Clustering, Step by Step
## DATA 78000 · AI and Machine Learning for Artists and Humanists

**Research question:** How does the language used in HasanAbi and Fox News comments change before, on, and after November 3, 2026?

This follows Professor Mandiberg's class demo on Iris: **load data → make numeric features → elbow method → fit K-Means → label rows → inspect centroids → visualize → interpret**.

The earlier production script lives at scripts/analyze.py and updates the public site. **This is a teaching version; it does not update the live site.**

Class demo: https://github.com/mandiberg/DATA-78000/tree/main/data_clustering

**Privacy:** Individual comment text stays in the ignored data/private/ folder. The notebook has no saved outputs. Do not commit executed notebooks that expose comments.

## Before starting — Run this notebook in Colab

The notebook now has a complete **YouTube comment collector built in**. Run the imports in **Step 1**, then edit the **Colab dataset builder** settings and run the collector cells before continuing to Step 2.

1. Open this notebook using [Google Colab](https://colab.research.google.com/github/jjwhite224/midtermdiscourseanalysis/blob/main/notebooks/kmeans_midterm_walkthrough.ipynb).
2. Enable the **YouTube Data API v3** in a Google Cloud project.
3. Add a Colab secret named `YOUTUBE_API_KEY` in the Secrets panel (key icon) and allow notebook access.
4. Edit the `VIDEO_SELECTION`, phase windows, sample size and sampling method. The configuration cell has editable values and Colab form controls.
5. Preview video metadata, collect the comments, review the six group counts, and then run all analysis steps.

Alternatively, if you already collected a private CSV locally, set `RUN_COLLECTION = False` and use the existing `comments.csv` loading cell at Step 2. You do **not** need to separately run the command-line collector.

Only aggregate results belong in the public repository. The Colab runtime is temporary and the raw comment text must remain private.

## Step 1 — Import the Python libraries

These tools replace the Iris dataset's built-in numerical measurements with numerical representations of comment text.

In [ ]:
import html
import re
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Imports complete.")

## Colab dataset builder — collect your own election comments

**You can now collect, modify, and review the dataset without leaving Colab.** The next cells sit **before Step 2** (loading the data for clustering).

1. **Edit the dataset specification:** choose channels, video IDs, time windows, sample sizes, and sampling method.
2. **Verify the videos** against official YouTube channel IDs and upload dates.
3. **Collect timestamp-filtered comments**, inspect the collection report and group counts, and save the CSV in the temporary Colab session.
4. Run **Step 2** and the rest of the clustering lesson on your new \`df\`.

**API key:** in Colab, open the **key icon → Secrets** and add \`YOUTUBE_API_KEY\`. Enable notebook access. Alternatively, the notebook will request a hidden key entry. Do **not** paste API keys into saved code cells.

**Privacy:** The resulting \`comments.csv\` includes comment texts and IDs and should remain private, outside GitHub. The collector does not save author names, handles, or channel memberships. Avoid publishing identifiable comment excerpts. Colab's temporary files may disappear when the runtime resets; download or move your private dataset if you need it again.

**Cost and scope:** This uses YouTube Data API v3. More videos and a larger page limit require more API requests and quota. This collector retrieves **top-level comments** only. A comment's *publication time* determines its phase—not just the video's upload time.

In [ ]:
# EDIT THIS CELL TO CONTROL HOW YOUR DATASET IS MADE.
# Set to False when you already have a local CSV or want to skip API access.
RUN_COLLECTION = True #@param {type:"boolean"}

# All windows use Eastern Time, with [start, end) boundaries.
TIMEZONE = "America/New_York"
PHASE_WINDOWS = {
    "before":   ("2026-10-27T00:00:00", "2026-11-03T00:00:00"),
    "election": ("2026-11-03T00:00:00", "2026-11-04T00:00:00"),
    "after":    ("2026-11-04T00:00:00", "2026-11-11T00:00:00"),
}

# Official channel IDs. Add more names + IDs when you expand to 8 channels.
CHANNEL_IDS = {
    "HasanAbi": "UCtoaZpBnrd0lhycxYJ4MNOQ",
    "Fox News": "UCXIJgqnII2ZOINSWNOGFThA"
}

# Each row is one video to collect. Change an ID to compare a different upload.
# The Fox pre-election ID was discovered by the first live run; consider a
# more election-poll-focused Fox video in your next experiment.
VIDEO_SELECTION = [
    {"channel": "HasanAbi", "phase": "before",   "video_id": "VLb58bb-sQ8", "enabled": True},
    {"channel": "HasanAbi", "phase": "election", "video_id": "EPhDfkvR1AQ", "enabled": True},
    {"channel": "HasanAbi", "phase": "after",    "video_id": "aO61Eyw-jEs", "enabled": True},
    {"channel": "Fox News", "phase": "before",   "video_id": "F6AZHs-Zy7Q", "enabled": True},
    {"channel": "Fox News", "phase": "election", "video_id": "1FBFCOrpE-o", "enabled": True},
    {"channel": "Fox News", "phase": "after",    "video_id": "Bxi-I9eFBxk", "enabled": True}
]

# Sampling choices. For the original pilot use target=150, minimum likes=0,
# minimum words=1, and hash sampling.
TARGET_PER_VIDEO = 150 #@param {type:"integer"}
MAX_PAGES_PER_VIDEO = 150 #@param {type:"integer"}
MIN_LIKES = 0 #@param {type:"integer"}
MIN_WORDS_AT_COLLECTION = 1 #@param {type:"integer"}
SAMPLING_METHOD = "hash" #@param ["hash", "recent"]
REQUIRE_VIDEO_UPLOAD_IN_PHASE = True
OUTPUT_DIR = Path("data/private")  # Colab: /content/data/private (temporary)

assert 1 <= TARGET_PER_VIDEO <= 1000
assert MAX_PAGES_PER_VIDEO >= 1
assert MIN_LIKES >= 0 and MIN_WORDS_AT_COLLECTION >= 1
assert SAMPLING_METHOD in ("hash", "recent")
print("Configured", sum(v["enabled"] for v in VIDEO_SELECTION),
      "videos; target =", TARGET_PER_VIDEO, "comments each.")

## Collection A — Secure authentication and video verification

The next cell reads your Colab secret (or prompts for hidden input). It checks the **actual uploader and publication timestamp** for every enabled video.

**Check the preview before collecting:** if a candidate is a short, off-topic upload, or doesn't match the research question, edit its ID in the configuration cell and rerun this step.

If any selected video fails verification, collection stops instead of silently mixing unrelated channels or periods.

In [ ]:
# YouTube Data API helpers; no extra pip package required.
import hashlib
import json
import os
import time
import getpass
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from zoneinfo import ZoneInfo

YOUTUBE_API_ROOT = "https://www.googleapis.com/youtube/v3/"

def read_youtube_key():
    # Prefer an environment variable or Colab's protected Secrets facility.
    key = os.environ.get("YOUTUBE_API_KEY", "").strip()
    if not key:
        try:
            from google.colab import userdata
            key = (userdata.get("YOUTUBE_API_KEY") or "").strip()
        except Exception:
            pass
    if not key:
        key = getpass.getpass("Paste YouTube Data API key (hidden): ").strip()
    if not key:
        raise ValueError("YouTube API key is required.")
    return key

def youtube_api(endpoint, params, key, retries=3):
    address = YOUTUBE_API_ROOT + endpoint + "?" + urlencode({**params, "key": key})
    for attempt in range(retries):
        try:
            with urlopen(Request(address, headers={
                "User-Agent": "MidtermDiscourseClassNotebook/1.0"
            }), timeout=40) as resp:
                return json.load(resp)
        except HTTPError as err:
            # Do not log a URL containing the secret API key.
            message = err.read().decode("utf-8", errors="replace")[:400].replace(key, "[REDACTED]")
            if err.code not in (429, 500, 502, 503) or attempt == retries - 1:
                raise RuntimeError(f"YouTube API error {err.code}: {message}") from None
        except URLError as err:
            if attempt == retries - 1:
                raise RuntimeError("YouTube API could not be reached.") from None
        time.sleep(2 ** attempt)

def utc_timestamp(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)

def phase_bounds(phase):
    zone = ZoneInfo(TIMEZONE)
    start, end = PHASE_WINDOWS[phase]
    return (
        datetime.fromisoformat(start).replace(tzinfo=zone).astimezone(timezone.utc),
        datetime.fromisoformat(end).replace(tzinfo=zone).astimezone(timezone.utc)
    )

def verify_selection(spec, key):
    channel, phase, video_id = spec["channel"], spec["phase"], spec["video_id"]
    if channel not in CHANNEL_IDS or phase not in PHASE_WINDOWS:
        raise ValueError(f"Unknown channel or election phase: {channel} / {phase}")
    if not isinstance(video_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError(f"Invalid YouTube video ID: {video_id}")
    res = youtube_api("videos", {"part": "snippet", "id": video_id}, key)
    if not res.get("items"):
        raise ValueError(f"YouTube video unavailable: {video_id}")
    details = res["items"][0]["snippet"]
    if details["channelId"] != CHANNEL_IDS[channel]:
        raise ValueError(f"{video_id} is not uploaded by the configured {channel} channel")
    start, end = phase_bounds(phase)
    published = utc_timestamp(details["publishedAt"])
    if REQUIRE_VIDEO_UPLOAD_IN_PHASE and not (start <= published < end):
        raise ValueError(f"{video_id} upload date is outside the {phase} window")
    return {
        "channel": channel, "phase": phase, "video_id": video_id,
        "title": details["title"],
        "uploaded_at_utc": details["publishedAt"],
        "url": "https://www.youtube.com/watch?v=" + video_id
    }

In [ ]:
# Run this cell after editing VIDEO_SELECTION.
VERIFIED_VIDEOS = []
if RUN_COLLECTION:
    _youtube_key = read_youtube_key()  # Kept in memory, never saved in metadata/CSV.
    enabled = [v for v in VIDEO_SELECTION if v.get("enabled", True)]
    if not enabled:
        raise ValueError("Enable at least one video in VIDEO_SELECTION.")
    if len(set(v["video_id"] for v in enabled)) != len(enabled):
        raise ValueError("A video ID appears more than once. Choose distinct videos.")
    for spec in enabled:
        meta = verify_selection(spec, _youtube_key)
        VERIFIED_VIDEOS.append(meta)
        print(meta["channel"], "|", meta["phase"], "|", meta["title"])
    display(pd.DataFrame(VERIFIED_VIDEOS)[
        ["channel", "phase", "video_id", "title", "uploaded_at_utc"]
    ])
    print("Video metadata verified. Confirm these choices before continuing.")
else:
    print("Collector skipped; Step 2 will load an existing private CSV.")

## Collection B — Gather comments and choose your sampling method

The API returns comments in **newest-first** order, up to 100 per page. We keep only comments whose own \`publishedAt\` falls inside the selected election window.

- **hash** (recommended): collects all available eligible comments up to the page cap and picks a deterministic pseudorandom subset. This avoids selecting only the most-liked or newest comments.
- **recent**: takes the latest eligible comments and is intentionally **recency-biased**.
- **MIN_LIKES** and **MIN_WORDS_AT_COLLECTION** are optional filters. Increasing them changes who/what is represented.
- **MAX_PAGES_PER_VIDEO** limits API calls. If a run stops at this cap, sampling is incomplete and possibly biased.

The collector **does not use stop words**. Those are applied later in clustering, so you can rerun preprocessing without paying to collect the same comments again.

Important: comments deleted or hidden before collection cannot be recovered, and a balanced sample of 150 per video is not representative of all YouTube users.

In [ ]:
def collect_one_video(meta, key):
    start, end = phase_bounds(meta["phase"])
    found = {}
    page_token = None
    pages = 0
    stopped_at_boundary = False
    while pages < MAX_PAGES_PER_VIDEO:
        params = {
            "part": "snippet", "videoId": meta["video_id"],
            "maxResults": 100, "order": "time", "textFormat": "plainText"
        }
        if page_token:
            params["pageToken"] = page_token
        payload = youtube_api("commentThreads", params, key)
        pages += 1
        for thread in payload.get("items", []):
            comment = thread["snippet"]["topLevelComment"]
            snippet = comment["snippet"]
            when = utc_timestamp(snippet["publishedAt"])  # NOT updatedAt
            if when < start:
                stopped_at_boundary = True
                continue
            if when >= end:
                continue
            body = (snippet.get("textDisplay") or "").strip()
            likes = int(snippet.get("likeCount") or 0)
            if not body or likes < MIN_LIKES or len(body.split()) < MIN_WORDS_AT_COLLECTION:
                continue
            cid = comment["id"]
            found[cid] = {
                "comment_id": cid,
                "video_id": meta["video_id"],
                "channel": meta["channel"],
                "phase": meta["phase"],
                "published_at_utc": snippet["publishedAt"],
                "text": body, "likes": likes
            }
        page_token = payload.get("nextPageToken")
        if stopped_at_boundary or not page_token:
            break

    eligible = list(found.values())
    if SAMPLING_METHOD == "hash":
        eligible.sort(key=lambda row: hashlib.sha256(
            (row["video_id"] + ":" + row["comment_id"]).encode("utf-8")
        ).hexdigest())
    elif SAMPLING_METHOD == "recent":
        eligible.sort(key=lambda row: row["published_at_utc"], reverse=True)
    sample = eligible[:TARGET_PER_VIDEO]
    complete = bool(stopped_at_boundary or not page_token)
    diagnostics = {
        "channel": meta["channel"], "phase": meta["phase"],
        "video_id": meta["video_id"],
        "eligible_scanned": len(eligible),
        "selected": len(sample),
        "pages_scanned": pages,
        "scan_complete": complete,
        "stop_reason": ("phase_boundary" if stopped_at_boundary else
                        "exhausted" if not page_token else "page_cap"),
        "under_target": len(sample) < TARGET_PER_VIDEO
    }
    return sample, diagnostics

In [ ]:
# Make the dataset in this Colab session. You can rerun this after changing
# a video ID, date, sampling method, or target.
if RUN_COLLECTION:
    collected = []
    reports = []
    for meta in VERIFIED_VIDEOS:
        try:
            rows, diagnosis = collect_one_video(meta, _youtube_key)
            collected.extend(rows)
            reports.append(diagnosis)
            print(f"{meta['channel']:9s} / {meta['phase']:8s}: "
                  f"{diagnosis['selected']} sampled from "
                  f"{diagnosis['eligible_scanned']} eligible; "
                  f"{diagnosis['stop_reason']}")
        except RuntimeError as exc:
            reports.append({
                "channel": meta["channel"], "phase": meta["phase"],
                "video_id": meta["video_id"], "status": "error",
                "error": str(exc)
            })
            print("WARNING: Collection failed for", meta["channel"], meta["phase"])
    if not collected:
        raise RuntimeError("No comments collected. Check the report and video selections.")

    df = pd.DataFrame(collected)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = OUTPUT_DIR / "comments.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")

    # A metadata-only record makes sampling choices reproducible.
    metadata = {
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
        "timezone": TIMEZONE, "phase_windows": PHASE_WINDOWS,
        "target_per_video": TARGET_PER_VIDEO,
        "max_pages_per_video": MAX_PAGES_PER_VIDEO,
        "min_likes": MIN_LIKES,
        "min_words_at_collection": MIN_WORDS_AT_COLLECTION,
        "sampling_method": SAMPLING_METHOD,
        "videos": VERIFIED_VIDEOS, "reports": reports,
        "total_comments": len(df)
    }
    (OUTPUT_DIR / "collection_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print("\nPrivate dataset saved:", csv_path.resolve())
    print("Collection metadata saved:", (OUTPUT_DIR / "collection_metadata.json").resolve())
    print("Raw comment texts and IDs have NOT been uploaded to GitHub.")
else:
    print("Collection was skipped; use Step 2 to load a saved CSV.")

## Collection C — Check your dataset before clustering

Use the following summary to catch missing videos, small groups or truncated API pagination. A dataset with fewer than the target number of comments is **not a failure**—document the shortfall.

**Privacy:** This diagnostic intentionally displays only counts and collection settings. It does not display comment bodies or usernames. A Colab runtime is temporary; you may manually download your CSV if you need to keep it, but do not commit it to a public repo.

In [ ]:
if RUN_COLLECTION:
    print("Rows collected:", len(df))
    print("\nChannel × election phase counts:")
    display(pd.crosstab(df["channel"], df["phase"]))
    display(pd.DataFrame(reports)[
        ["channel", "phase", "selected", "eligible_scanned",
         "pages_scanned", "scan_complete", "stop_reason", "under_target"]
    ].fillna("error"))
    print("\nTo download the private CSV in Colab, uncomment these lines:")
    print("# from google.colab import files")
    print("# files.download(str(csv_path))")
    print("Keep the CSV private; it includes comment text and IDs.")

## Step 2 — Load the dataset you just created

If you ran the **Colab collector**, we reuse the \`df\` already in memory. Otherwise, this cell looks for a private \`comments.csv\` in your notebook directory or \`data/private/\`.

**Before clustering:** check the six channel/period groups. Adding channels and videos is fine: all rows will be analyzed using the same shared TF-IDF vocabulary, but you must also update the comparison chart in Step 9 to include the new channel names.

In [ ]:
# Prefer the DataFrame created above; alternatively load a private CSV.
if "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    df = df.copy()
    print("Using in-memory Colab dataset collected above.")
else:
    paths = [
        Path("data/private/comments.csv"),
        Path("../data/private/comments.csv"),
        Path("comments.csv"),
    ]
    matches = [p for p in paths if p.exists()]
    if not matches:
        raise FileNotFoundError(
            "No comments available. Set RUN_COLLECTION=True and run the "
            "collector cells above, or upload your private comments.csv."
        )
    df = pd.read_csv(matches[0])
    print("Loaded existing private CSV:", matches[0])

required = {"comment_id", "video_id", "channel", "phase", "published_at_utc", "text"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV lacks columns: {sorted(missing)}")
df = df.drop_duplicates(subset="comment_id").dropna(subset=["text"]).copy()
print("Unique comments:", len(df))
print(pd.crosstab(df["channel"], df["phase"]).to_string())

## Step 3 — Clean the text

We remove links, decode HTML escapes, and lowercase. Like the existing script, comments shorter than three words are excluded. Write down how many are excluded: that choice can change the results.

In [ ]:
def clean_text(value):
    value = html.unescape(str(value)).lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

df["text_clean"] = df["text"].apply(clean_text)
df["word_count"] = df["text_clean"].str.split().str.len()
eligible = df.loc[df["word_count"] >= 3].copy().reset_index(drop=True)
print("Eligible comments:", len(eligible))
print("Excluded short comments:", len(df) - len(eligible))
print(pd.crosstab(eligible["channel"], eligible["phase"]).to_string())
if len(eligible) < 12:
    raise ValueError("At least 12 eligible comments are needed.")

## Step 3A — Standard stop words + your own exclusion list

This adapts Mandiberg's [custom stop words pattern](https://github.com/mandiberg/DATA-78000/blob/main/topic_model/custom_stop_words_pattern.py): start with an existing stop-word set, then create a **union** with your own words. His example uses Gensim's `STOPWORDS`; because **our clustering uses scikit-learn's TF-IDF**, we use `ENGLISH_STOP_WORDS` instead.

**Why?** If `hasan`, `fox`, `like`, or `thumbnail` dominate, K-Means may group comments by channel or casual phrasing instead of substantive election discussions.

Our lists have different purposes:

- **STANDARD_STOP_WORDS:** common English words automatically provided by scikit-learn.
- **CUSTOM_STOP_WORDS:** additional recurring filler, branding and platform terms to remove in this experiment. Edit this set yourself.
- **REVIEW_ONLY_WORDS:** potentially off-topic or meme words found in the first pilot, **off by default** because some could be meaningful political reactions. Turn on only after inspecting comments.

**Do not automatically remove** `vote`, `voted`, `midterm`, `election`, `democrats`, `republicans`, `trump`, `blue`, `iran`, `texas`, etc. Those may be exactly the concepts you want to analyze.

Compare the before/after table **before** rerunning the elbow and clusters. This is a researcher choice, not objective cleaning.

In [ ]:
# Mandiberg's custom stop-word approach, adapted to scikit-learn:
# 1. Start with a standard stop-word set.
STANDARD_STOP_WORDS = set(ENGLISH_STOP_WORDS)

# 2. Add your own words. Change this list based on what you observe.
CUSTOM_STOP_WORDS = {
    "hasan", "hasanabi", "fox", "youtube", "channel", "video",
    "subscribe", "subscribed", "thumbnail",
    "lol", "lmao", "yeah", "gonna", "wanna", "im", "ive",
    "dont", "doesnt", "didnt", "youre", "thats", "grokspx"
}

# 3. These might be topical references, jokes, or actual research findings.
# Don't remove them unless a private review justifies it.
REMOVE_REVIEW_WORDS = False
REVIEW_ONLY_WORDS = {"rachel", "outfit", "collab", "bluke", "blexas"}

# 4. Union the sets, as in the professor's custom_stop_words_pattern.py.
STOP_WORDS = STANDARD_STOP_WORDS.union(CUSTOM_STOP_WORDS)
if REMOVE_REVIEW_WORDS:
    STOP_WORDS = STOP_WORDS.union(REVIEW_ONLY_WORDS)

# 5. Show what will change. A token is a 2+-character word-like piece.
token_pattern = re.compile(r"(?u)\b\w\w+\b")
def words_in(text):
    return token_pattern.findall(text)

before_counts = Counter(
    word for comment in eligible["text_clean"] for word in words_in(comment)
)
after_counts = Counter({
    word: count for word, count in before_counts.items() if word not in STOP_WORDS
})
removed = sum(before_counts.values()) - sum(after_counts.values())

print("English stop words:", len(STANDARD_STOP_WORDS))
print("Custom words:", sorted(CUSTOM_STOP_WORDS))
print("Removed token occurrences:", removed,
      f"of {sum(before_counts.values())} ({removed / max(1, sum(before_counts.values())):.1%})")
print("\nMost frequent words BEFORE filtering:")
print(before_counts.most_common(15))
print("\nMost frequent words AFTER filtering:")
print(after_counts.most_common(15))

# Teaching example: show a made-up sentence, not someone's private comment.
example = "Hasan just talked about the Senate election; I voted today!"
example_tokens = words_in(clean_text(example))
print("\nSynthetic example before:", example_tokens)
print("Synthetic example after:",
      [token for token in example_tokens if token not in STOP_WORDS])

## Step 4 — Turn remaining words into numerical features (TF-IDF)

Now we pass the **combined** stop-word list into `TfidfVectorizer(stop_words=sorted(STOP_WORDS))`. The vectorizer filters these tokens *before* building unigrams and bigrams; removing a word from the vocabulary prevents it from influencing the clustering.

`min_df=2` also removes terms appearing in fewer than two comments. `max_df=0.90` removes terms that appear in over 90% of comments. **These thresholds are separate from the stop-word list.**

We fit **one shared vectorizer across both channels and every period**. We also exclude all-zero vectors (comments with no remaining TF-IDF features), reporting how many; otherwise those comments can distort clusters.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words=sorted(STOP_WORDS),
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=3000
)
X = vectorizer.fit_transform(eligible["text_clean"])
feature_names = vectorizer.get_feature_names_out()
print("Comments x features:", X.shape)
print("Example features:", list(feature_names[:15]))
print("Custom excluded words remaining as features:",
      sorted(set(CUSTOM_STOP_WORDS).intersection(feature_names)))
# Comments containing only removed words/no retained features become all-zero vectors.
# Exclude them from K-Means but keep them in df for sampling diagnostics.
nonzero_mask = X.getnnz(axis=1) > 0
zero_vector_count = int((~nonzero_mask).sum())
print("Additional comments with zero TF-IDF features:", zero_vector_count)
if zero_vector_count:
    eligible = eligible.loc[nonzero_mask].reset_index(drop=True)
    X = X[nonzero_mask]
print("Comments remaining for K-Means:", X.shape[0])
if X.shape[0] < 12:
    raise ValueError("Too few comments remaining after stop-word/TF-IDF filtering.")
if X.shape[1] < 2:
    raise ValueError("Not enough distinct TF-IDF features.")

## Step 5 — Make the elbow plot

This closely follows the class Iris demo. Inertia measures within-cluster squared distances. It decreases as k increases; look for where improvements begin to level out. The supplemental cosine-silhouette value is a *different* diagnostic, not the objective K-Means optimizes. Neither proves a cluster is meaningful.

In [ ]:
k_values = list(range(1, min(10, len(eligible) - 1) + 1))
inertia = []
silhouette = []
for k in k_values:
    trial = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    trial_labels = trial.fit_predict(X)
    inertia.append(trial.inertia_)
    if k >= 2 and len(set(trial_labels)) >= 2:
        value = silhouette_score(
            X, trial_labels, metric="cosine",
            sample_size=min(400, X.shape[0]), random_state=RANDOM_SEED
        )
    else:
        value = np.nan
    silhouette.append(value)
scores = pd.DataFrame({"k": k_values, "inertia": inertia,
                       "cosine_silhouette": silhouette})
print(scores.round(3).to_string(index=False))
plt.figure(figsize=(8, 4))
plt.plot(k_values, inertia, marker="o", linestyle="--")
plt.xticks(k_values)
plt.xlabel("Number of clusters, k")
plt.ylabel("Inertia")
plt.title("Elbow method: midterm comments")
plt.grid(alpha=0.2)
plt.show()

## Step 6 — Choose k, fit K-Means and attach labels

The original script chose k=6 automatically based on dataset size. For this classroom exercise, **start with 4** and compare it to 3, 5, and 6. Justify your final choice using the elbow plot **and** human interpretation. Cluster IDs are arbitrary, not ideology labels.

In [ ]:
CHOSEN_K = 4   # Change this after reading the plot and cluster contents.
final_model = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_SEED, n_init=10)
cluster_labels = final_model.fit_predict(X)
eligible["cluster"] = cluster_labels   # Like the class assign-back-to-dataframe example.
print("Final inertia:", round(final_model.inertia_, 2))
print("Cluster sizes:")
print(eligible["cluster"].value_counts().sort_index().to_string())

## Step 7 — Examine centroids, like the Iris exercise

A centroid here is a vector of TF-IDF weights, not flower measurements. Print its highest-weight words. Give clusters descriptive labels **only after** checking their actual content.

In [ ]:
centroids = final_model.cluster_centers_
rows = []
for cluster_id in range(CHOSEN_K):
    indices = np.argsort(centroids[cluster_id])[::-1][:10]
    rows.append({
        "cluster": cluster_id,
        "n_comments": int((eligible["cluster"] == cluster_id).sum()),
        "top_words": ", ".join(feature_names[i] for i in indices)
    })
centroid_table = pd.DataFrame(rows)
print(centroid_table.to_string(index=False))

## Step 8 — Inspect examples privately

This mirrors the final stage of the Iris demo that prints points belonging to each cluster. Here we identify five comments closest to each centroid. Keep the text private: set SHOW_PRIVATE_EXAMPLES to True only in your own session, and remove outputs before sharing.

In [ ]:
SHOW_PRIVATE_EXAMPLES = False
distances = final_model.transform(X)
for cluster_id in range(CHOSEN_K):
    members = np.where(cluster_labels == cluster_id)[0]
    nearest = sorted(members, key=lambda i: distances[i, cluster_id])[:5]
    print(f"\nCluster {cluster_id}: {len(members)} eligible comments")
    print(centroid_table.loc[cluster_id, "top_words"])
    if SHOW_PRIVATE_EXAMPLES:
        for row_index in nearest:
            item = eligible.iloc[row_index]
            print(f"  {item['channel']} / {item['phase']}: {item['text'][:180]}")
    else:
        print("  Examples hidden; enable private review to assess interpretability.")

## Step 9 — Compare group percentages rather than raw counts

Now use the channel and phase metadata **after** clustering. Percentage denominators are the eligible comments per channel/phase, not necessarily 150, because the short-comment exclusion removed some rows. This is the core test of the research question.

In [ ]:
# Dynamically use all channel names and phases found in the new dataset.
channels = list(dict.fromkeys(eligible["channel"].tolist()))
phases = [p for p in PHASE_WINDOWS if p in set(eligible["phase"])] if "PHASE_WINDOWS" in globals() else sorted(eligible["phase"].unique())
index = pd.MultiIndex.from_product([channels, phases], names=["channel", "phase"])
counts = pd.crosstab(
    index=[eligible["channel"], eligible["phase"]],
    columns=eligible["cluster"]
).reindex(index, fill_value=0).reindex(columns=range(CHOSEN_K), fill_value=0)
percentages = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
print("Percent of eligible comments per group:")
print(percentages.round(1).to_string())
plt.figure(figsize=(9, 5))
plt.imshow(percentages.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
plt.colorbar(label="% of eligible group")
plt.xticks(range(CHOSEN_K), [f"Cluster {i}" for i in range(CHOSEN_K)])
plt.yticks(range(len(index)), [f"{c} / {p}" for c, p in index])
plt.title("How cluster prevalence changes over the election")
plt.tight_layout()
plt.show()

## Step 10 — Plot two dimensions, as in the Iris demo

The demo plots two flower measurements. TF-IDF has thousands of dimensions, so here we project comments and centroids into two dimensions with TruncatedSVD. The projection loses information; it is **not** the space in which K-Means was trained.

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=RANDOM_SEED)
points_2d = svd.fit_transform(X)
centers_2d = svd.transform(final_model.cluster_centers_)
plt.figure(figsize=(8, 6))
plt.scatter(points_2d[:, 0], points_2d[:, 1],
            c=cluster_labels, s=20, alpha=0.5, cmap="viridis")
plt.scatter(centers_2d[:, 0], centers_2d[:, 1],
            marker="X", c="red", s=150, label="Projected centroids")
plt.title("Comment clusters in a 2D projection")
plt.xlabel("SVD axis 1")
plt.ylabel("SVD axis 2")
plt.legend()
plt.show()
print("Fraction of variance explained by 2 axes:",
      f"{svd.explained_variance_ratio_.sum():.1%}")

## Step 11 — Repeat with other cluster counts

K-Means always assigns points to clusters, even when those clusters are incoherent. Inspect whether one large catch-all cluster dominates, as it did in the original six-cluster pilot.

In [ ]:
comparison = []
for k in [3, 4, 5, 6]:
    if k >= len(eligible):
        continue
    trial = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    assignments = trial.fit_predict(X)
    sizes = pd.Series(assignments).value_counts()
    comparison.append({"k": k, "inertia": round(trial.inertia_, 2),
                       "largest_cluster_pct": round(sizes.max() / len(eligible) * 100, 1),
                       "smallest_cluster_n": int(sizes.min())})
print(pd.DataFrame(comparison).to_string(index=False))

## Step 12 — Interpretation / reflection (write this in your own words)

1. Compare vocabulary BEFORE vs AFTER standard/custom stop-word removal. Which words were removed, and which politically meaningful words did you preserve?
2. What changes in topic prevalence before, on, and after Election Day?
3. Which **specific words** characterize each cluster?
4. Does the elbow curve support your chosen k? Which k gives the clearest groups?
5. How many comments were excluded as short or zero-vector? Do rates vary by channel or period?
6. Did any cluster largely capture generic language, memes, or unrelated news instead of election narratives?
7. Did video selection or differing comment moderation create an alternative explanation?
8. Rerun with REMOVE_REVIEW_WORDS=True only if justified. Compare both k and cluster interpretations. What would change before expanding to 24 videos?

The previous k=6 run assigned **512 of 831** model-eligible comments to one generic cluster (61.6%). Treat this as an interpretability warning, **not as evidence** of an ideological group.

**Deliverable for class:** an elbow figure, a cluster-size table, a centroid/top-words table, a normalized channel-period heatmap, a projected scatterplot, and a discussion of meaningfulness/limitations.

Keep the existing scripts/analyze.py for the deployed site; the notebook is intentionally independent.